# 📦 Lector de Archivos Parquet — Versión Completa
Este notebook permite cargar, inspeccionar y explorar archivos `.parquet` con pandas, pyarrow y plotly.

In [ ]:
# Instalación de dependencias (ejecutar solo si no están instaladas)
# !pip install pandas pyarrow plotly

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print(f'pandas  : {pd.__version__}')
print(f'pyarrow : {pa.__version__}')

## 1. Cargar el archivo

In [ ]:
# Cambia esta ruta a la ubicación de tu archivo .parquet
RUTA_PARQUET = 'tu_archivo.parquet'

df = pd.read_parquet(RUTA_PARQUET)
tabla = pq.read_table(RUTA_PARQUET)
meta = pq.read_metadata(RUTA_PARQUET)

print(f'✅ Archivo cargado correctamente')
print(f'   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')

## 2. Vista previa de los datos

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

## 3. Inspección del esquema y tipos

In [ ]:
# Tipos de datos de cada columna
print('=== Tipos de datos ===')
print(df.dtypes)
print()

# Información general
print('=== Información general ===')
df.info()

In [ ]:
# Esquema pyarrow más detallado
print('=== Esquema PyArrow ===')
print(tabla.schema)

In [ ]:
# Tabla resumen de tipos + nulos por columna
pa_tipos = {field.name: str(field.type) for field in tabla.schema}
resumen_tipos = pd.DataFrame({
    'Tipo Pandas'  : df.dtypes.astype(str),
    'Tipo PyArrow' : pa_tipos,
    'No Nulos'     : df.count(),
    'Nulos'        : df.isnull().sum(),
    '% Nulos'      : (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos'       : df.nunique(),
})
display(resumen_tipos)

## 4. Estadísticas descriptivas

In [ ]:
df.describe(include='all')

In [ ]:
# Distribución de tipos de columna
conteo = df.dtypes.astype(str).value_counts().reset_index()
conteo.columns = ['Tipo', 'Cantidad']
fig = px.pie(
    conteo, values='Cantidad', names='Tipo',
    title='Distribución de tipos de columna',
    hole=0.4, color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textinfo='label+percent+value')
fig.show()

## 5. Valores nulos

In [ ]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

if resumen_nulos.empty:
    print('✅ No hay valores nulos en el dataset')
else:
    print('⚠️ Columnas con valores nulos:')
    display(resumen_nulos)

In [ ]:
# Gráfico de barras: % de nulos por columna
if not resumen_nulos.empty:
    cols_nulas = nulos[nulos > 0]
    pct = (cols_nulas / len(df) * 100).round(2)
    fig = px.bar(
        x=cols_nulas.index, y=pct.values,
        labels={'x': 'Columna', 'y': '% Nulos'},
        title=f'{len(cols_nulas)} columnas con valores nulos',
        color=pct.values, color_continuous_scale='Reds', text=pct.values,
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

In [ ]:
# Heatmap de nulos (rojo = nulo) — muestra hasta 5 000 filas
if not resumen_nulos.empty:
    cols_nulas = nulos[nulos > 0]
    muestra = df[cols_nulas.index].isnull().head(min(len(df), 5_000))
    fig = px.imshow(
        muestra.T.astype(int),
        color_continuous_scale=['white', 'crimson'],
        aspect='auto',
        title='Heatmap de nulos — muestra hasta 5 000 filas',
        labels={'color': 'Nulo'},
    )
    fig.update_coloraxes(showscale=False)
    fig.show()

## 6. Distribuciones numéricas

In [ ]:
# Histogramas de columnas numéricas
MAX_HIST_COLS = 9  # ajusta según cuántas quieras ver

num_df = df.select_dtypes(include='number')
cols_num = num_df.columns[:MAX_HIST_COLS]
n = len(cols_num)

if n == 0:
    print('No hay columnas numéricas')
else:
    ncols_g = min(3, n)
    nrows_g = (n + ncols_g - 1) // ncols_g
    fig = make_subplots(rows=nrows_g, cols=ncols_g, subplot_titles=list(cols_num))
    for i, col in enumerate(cols_num):
        r, c = divmod(i, ncols_g)
        fig.add_trace(
            go.Histogram(x=num_df[col].dropna(), name=col,
                         marker_color='steelblue', nbinsx=30, showlegend=False),
            row=r + 1, col=c + 1,
        )
    fig.update_layout(height=300 * nrows_g, title_text='Histogramas')
    fig.show()

In [ ]:
# Box plots — cambia COL_BOX por la columna que quieras explorar
COL_BOX = num_df.columns[0]  # primera columna numérica por defecto

fig = px.box(df, y=COL_BOX, title=f'Box plot: {COL_BOX}',
             color_discrete_sequence=['steelblue'])
fig.show()

## 7. Frecuencia de categorías

In [ ]:
# Top valores de columnas categóricas
TOP_CATS = 10  # ajusta cuántos valores mostrar

cat_df = df.select_dtypes(exclude='number')
cols_cat = [c for c in cat_df.columns if df[c].nunique() <= 200]

if not cols_cat:
    print('No hay columnas categóricas con ≤ 200 valores únicos')
else:
    print('Columnas categóricas disponibles:')
    for c in cols_cat:
        print(f'  {c}  ({df[c].nunique()} únicos)')

In [ ]:
# Cambia COL_CAT por la columna que quieras explorar
COL_CAT = cols_cat[0]  # primera categórica por defecto

vc = df[COL_CAT].value_counts().head(TOP_CATS).reset_index()
vc.columns = ['Valor', 'Frecuencia']

fig = px.bar(
    vc, x='Frecuencia', y='Valor', orientation='h',
    title=f'Top {TOP_CATS} valores: {COL_CAT}',
    color='Frecuencia', color_continuous_scale='Greens', text='Frecuencia',
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

print(f'Valores únicos en {COL_CAT}: {int(df[COL_CAT].nunique())}')

## 8. Mapa de correlaciones

In [ ]:
# Método: 'pearson', 'spearman' o 'kendall'
METODO = 'pearson'

num_corr = df.select_dtypes(include='number')
if num_corr.shape[1] < 2:
    print('Se necesitan al menos 2 columnas numéricas para correlaciones')
else:
    corr = num_corr.corr(method=METODO)
    fig = px.imshow(
        corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
        text_auto='.2f', aspect='auto',
        title=f'Mapa de correlaciones ({METODO.capitalize()})',
    )
    fig.update_layout(height=600)
    fig.show()

## 9. Metadatos del archivo Parquet

In [ ]:
print(f'Número de grupos de filas : {meta.num_row_groups}')
print(f'Total de filas            : {meta.num_rows:,}')
print(f'Total de columnas         : {meta.num_columns}')
print(f'Tamaño serializado        : {meta.serialized_size:,} bytes')
print(f'Formato                   : {meta.format_version}')
print()

for i in range(meta.num_row_groups):
    rg = meta.row_group(i)
    print(f'  Grupo {i}: {rg.num_rows:,} filas, {rg.total_byte_size:,} bytes ({round(rg.total_byte_size/1024,1)} KB)')

## 10. Filtros y lectura parcial (eficiente para archivos grandes)

In [ ]:
# Leer solo columnas específicas
# columnas_deseadas = ['col1', 'col2']  # Ajusta los nombres
# df_parcial = pd.read_parquet(RUTA_PARQUET, columns=columnas_deseadas)

# Leer con filtro de filas (usando pyarrow)
# filtros = [('columna', '=', 'valor')]  # Ajusta la condición
# df_filtrado = pd.read_parquet(RUTA_PARQUET, filters=filtros)